### Author:: \<Mohammad_Amin_Kiani\>
ID::     \<4043644008\>

In [1]:
# =========================================================
# 1) Imports
# N-gram music LM assignment
# =========================================================

"""
What to do:
1) loads music.csv
2) normalizes notes optionally (ignore b/k modifiers, keep uppercase notes)
3) trains unigram / bigram / trigram Lidstone language models
4) tunes alpha by leave-one-out validation
5) computes perplexity
6) generates completions for the 5 incomplete prompts
7) repeats the whole experiment with modifiers ignored
8) optionally uses the dastgah column as a conditioning token

Important note:
- Generation is done with a fixed target length (10 missing notes).
- To avoid degenerate loops like C C C C..., a small repeated-token blocking
  heuristic is included in beam search. This makes the filled sequences more
  useful for the assignment, while still being a pure n-gram decoder.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import math
import re
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd


In [2]:
# =========================================================
# 2) Configuration
# =========================================================

PROMPTS = [
    ["S", "S", "S", "S"],
    ["G", "C", "G", "C"],
    ["C", "B", "C", "D"],
    ["A", "Bb", "C", "D"],
    ["A", "S", "B", "Db"],
]
NGRAM_ORDERS = [1, 2, 3]
ALPHA_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]
TARGET_COMPLETION_LEN = 10   # the missing part in each sequence
BEAM_WIDTH = 8
RANDOM_SEED = 42

### Dataset

In [3]:
# =========================================================
# 3) Load Dataset
# =========================================================

CSV_PATH = "/content/music.csv"


# Run1 :

### Preprocessing

In [ ]:
# =========================
# 4) Preprocessing
# =========================

def normalize_note(token: str, ignore_modifiers: bool = False) -> str:
    """
    Normalize a token from the note column.

    - Keeps 'S' as silence.
    - When ignore_modifiers=True:
        * uppercases the token
        * strips trailing 'b' or 'k' modifier if present
          (examples: Ab -> A, Bb -> B, Ak -> A, Bk -> B)
    """
    t = str(token).strip()
    if not t:
        return t
    if ignore_modifiers:
        if t.lower() == "s":
            return "S"
        t = t.upper()
        if len(t) > 1 and t[-1] in {"B", "K"} and t[0] in "ABCDEFG":
            t = t[0]
    return t


def tokenize_notes(note_string: str, ignore_modifiers: bool = False) -> List[str]:
    return [normalize_note(tok, ignore_modifiers) for tok in str(note_string).split() if tok.strip()]


def load_music_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "note" not in df.columns or "dastgah" not in df.columns or "name" not in df.columns:
        raise ValueError(f"Unexpected columns in {path}. Expected name, note, dastgah.")
    df["note"] = df["note"].astype(str).str.strip()
    df["dastgah"] = df["dastgah"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    return df


In [ ]:
# =========================
# 5) N-gram LM with Lidstone smoothing
# =========================

class NGramLM:
    def __init__(self, n: int, alpha: float = 1.0):
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.alpha = float(alpha)
        self.counts: Counter = Counter()
        self.context_counts: Counter = Counter()
        self.vocab: List[str] = []

    def fit(self, sequences: Sequence[Sequence[str]]) -> "NGramLM":
        self.counts = Counter()
        self.context_counts = Counter()
        vocab = set()

        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            vocab.update(padded)
            for i in range(len(padded) - self.n + 1):
                ngram = tuple(padded[i : i + self.n])
                context = ngram[:-1]
                word = ngram[-1]
                self.counts[ngram] += 1
                self.context_counts[context] += 1

        # Never generate the start token.
        vocab.discard("<s>")
        self.vocab = sorted(vocab)
        return self

    def prob(self, word: str, context: Sequence[str]) -> float:
        context = tuple(context)
        if self.n == 1:
            context = ()
        else:
            if len(context) < self.n - 1:
                context = ("<s>",) * (self.n - 1 - len(context)) + context
            elif len(context) > self.n - 1:
                context = context[-(self.n - 1) :]
        V = len(self.vocab)
        c_ngram = self.counts[context + (word,)]
        c_context = self.context_counts[context]
        return (c_ngram + self.alpha) / (c_context + self.alpha * V)

    def perplexity(self, sequences: Sequence[Sequence[str]]) -> float:
        total_logprob = 0.0
        total_tokens = 0
        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            total_tokens += len(seq) + 1  # include EOS
            for i in range(self.n - 1, len(padded)):
                context = tuple(padded[i - self.n + 1 : i]) if self.n > 1 else ()
                total_logprob += math.log(self.prob(padded[i], context))
        return math.exp(-total_logprob / total_tokens)

    def next_distribution(self, context: Sequence[str]) -> List[Tuple[str, float]]:
        context = list(context)
        if self.n > 1:
            if len(context) < self.n - 1:
                context = ["<s>"] * (self.n - 1 - len(context)) + context
            else:
                context = context[-(self.n - 1) :]
        dist = [(w, self.prob(w, context)) for w in self.vocab if w != "<s>"]
        dist.sort(key=lambda x: x[1], reverse=True)
        return dist

    def predict_next(self, context: Sequence[str]) -> str:
        return self.next_distribution(context)[0][0]

    def greedy_generate(self, prefix: Sequence[str], max_new_tokens: int = 10) -> List[str]:
        seq = list(prefix)
        for _ in range(max_new_tokens):
            nxt = self.predict_next(seq[-(self.n - 1) :] if self.n > 1 else [])
            if nxt == "</s>":
                break
            seq.append(nxt)
        return seq

    def beam_generate(
        self,
        prefix: Sequence[str],
        max_new_tokens: int = 10,
        beam_width: int = 5,
    ) -> List[str]:
        beams: List[Tuple[float, List[str]]] = [(0.0, list(prefix))]

        for _ in range(max_new_tokens):
            candidates: List[Tuple[float, List[str]]] = []
            for logp, seq in beams:
                dist = self.next_distribution(seq[-(self.n - 1) :] if self.n > 1 else [])
                for word, p in dist[:beam_width]:
                    new_seq = seq + [word]
                    new_logp = logp + math.log(p)
                    if word == "</s>":
                        candidates.append((new_logp, new_seq))
                    else:
                        candidates.append((new_logp, new_seq))

            if not candidates:
                break

            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]

        # Pick the best candidate. Trim EOS if present.
        best = max(beams, key=lambda x: x[0])[1]
        if "</s>" in best:
            best = best[: best.index("</s>")]
        return best


def tune_alpha(
    sequences: Sequence[Sequence[str]],
    n: int,
    alpha_grid: Sequence[float],
) -> Tuple[float, Dict[float, float]]:
    """
    Leave-one-out validation on sentence-perplexity.
    Returns the best alpha and a dict of mean validation perplexities.
    """
    scores: Dict[float, float] = {}
    for alpha in alpha_grid:
        perps = []
        for i in range(len(sequences)):
            train = list(sequences[:i]) + list(sequences[i + 1 :])
            val = [sequences[i]]
            lm = NGramLM(n=n, alpha=alpha).fit(train)
            perps.append(lm.perplexity(val))
        scores[alpha] = float(np.mean(perps))
    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores


def build_sequences(df: pd.DataFrame, ignore_modifiers: bool = False, use_dastgah: bool = False) -> List[List[str]]:
    sequences = []
    for _, row in df.iterrows():
        seq = tokenize_notes(row["note"], ignore_modifiers=ignore_modifiers)
        if use_dastgah:
            # Condition the LM on dastgah by prepending a special token.
            seq = [f"__DASTGAH_{normalize_note(row['dastgah'], ignore_modifiers=False)}__"] + seq
        sequences.append(seq)
    return sequences


def train_and_report(df: pd.DataFrame, ignore_modifiers: bool = False, use_dastgah: bool = False):
    sequences = build_sequences(df, ignore_modifiers=ignore_modifiers, use_dastgah=use_dastgah)
    print("=" * 80)
    print(f"ignore_modifiers={ignore_modifiers} | use_dastgah={use_dastgah}")
    print(f"Number of pieces: {len(sequences)}")
    vocab = sorted(set(tok for seq in sequences for tok in seq))
    print(f"Vocabulary size: {len(vocab)}")
    print()

    best_by_n = {}
    all_models = {}

    for n in NGRAM_ORDERS:
        best_alpha, alpha_scores = tune_alpha(sequences, n, ALPHA_GRID)
        best_by_n[n] = {"alpha": best_alpha, "scores": alpha_scores}
        model = NGramLM(n=n, alpha=best_alpha).fit(sequences)
        all_models[n] = model

        print(f"[n={n}] best alpha = {best_alpha}")
        print("Validation perplexity by alpha:")
        for a in ALPHA_GRID:
            print(f"  alpha={a:>4}: {alpha_scores[a]:.4f}")
        print(f"Train-set perplexity with best alpha: {model.perplexity(sequences):.4f}")
        print()

    # Also report required comparison values alpha=1 and alpha=0.5.
    print("Perplexity for required alpha values (1.0 and 0.5):")
    for n in NGRAM_ORDERS:
        p1 = NGramLM(n=n, alpha=1.0).fit(sequences).perplexity(sequences)
        p05 = NGramLM(n=n, alpha=0.5).fit(sequences).perplexity(sequences)
        print(f"  n={n}: alpha=1.0 -> {p1:.4f}, alpha=0.5 -> {p05:.4f}")
    print()

    # Generate completions with the best-performing n-gram order on validation.
    best_n = min(best_by_n, key=lambda n: best_by_n[n]["scores"][best_by_n[n]["alpha"]])
    best_alpha = best_by_n[best_n]["alpha"]
    best_model = all_models[best_n]
    print(f"Selected model for generation: n={best_n}, alpha={best_alpha}")
    print()

    for i, prefix in enumerate(PROMPTS, 1):
        if use_dastgah:
            # If you want to generate with dastgah-conditioning, prepend a chosen dastgah token here.
            # For the exercise, the prompts do not specify dastgah, so this is left as a demo.
            pass
        generated = best_model.beam_generate(prefix, max_new_tokens=10, beam_width=5)
        print(f"Prompt {i}: {' '.join(prefix)}")
        print(f"Generated: {' '.join(generated)}")
        print()

    return best_by_n, all_models



### Run

In [ ]:
def main():
    df = load_music_csv(CSV_PATH)

    # Main run: as-is
    train_and_report(df, ignore_modifiers=False, use_dastgah=False)

    # Second run: ignore b/k modifiers and uppercase everything.
    train_and_report(df, ignore_modifiers=True, use_dastgah=False)

    # Optional third run: use dastgah as a conditioning token.
    # to test the idea discussed in the written answer.
    train_and_report(df, ignore_modifiers=True, use_dastgah=True)


if __name__ == "__main__":
    main()

ignore_modifiers=False | use_dastgah=False
Number of pieces: 41
Vocabulary size: 14

[n=1] best alpha = 2.0
Validation perplexity by alpha:
  alpha= 0.1: 12.0478
  alpha= 0.2: 12.0356
  alpha= 0.3: 12.0264
  alpha= 0.4: 12.0186
  alpha= 0.5: 12.0115
  alpha=0.75: 11.9960
  alpha= 1.0: 11.9825
  alpha= 1.5: 11.9591
  alpha= 2.0: 11.9392
Train-set perplexity with best alpha: 10.8379

[n=2] best alpha = 1.0
Validation perplexity by alpha:
  alpha= 0.1: 9.7932
  alpha= 0.2: 9.4689
  alpha= 0.3: 9.3300
  alpha= 0.4: 9.2549
  alpha= 0.5: 9.2110
  alpha=0.75: 9.1664
  alpha= 1.0: 9.1658
  alpha= 1.5: 9.2167
  alpha= 2.0: 9.2954
Train-set perplexity with best alpha: 7.5064

[n=3] best alpha = 0.5
Validation perplexity by alpha:
  alpha= 0.1: 11.9744
  alpha= 0.2: 10.7137
  alpha= 0.3: 10.2807
  alpha= 0.4: 10.0973
  alpha= 0.5: 10.0218
  alpha=0.75: 10.0239
  alpha= 1.0: 10.1261
  alpha= 1.5: 10.3959
  alpha= 2.0: 10.6639
Train-set perplexity with best alpha: 5.5472

Perplexity for required al

# Run2 : (Better)

In [ ]:
# ---------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------
def normalize_note(token: str, ignore_modifiers: bool = False) -> str:
    """
    Normalize a note token.

    - Keeps 'S' as silence.
    - When ignore_modifiers=True:
        * uppercases the token
        * strips trailing 'b' or 'k' modifier if present
          (examples: Ab -> A, Bb -> B, Ak -> A, Bk -> B)
    """
    t = str(token).strip()
    if not t:
        return t

    if ignore_modifiers:
        if t.lower() == "s":
            return "S"
        t = t.upper()
        if len(t) > 1 and t[-1] in {"B", "K"} and t[0] in "ABCDEFG":
            t = t[0]
    return t


def tokenize_notes(note_string: str, ignore_modifiers: bool = False) -> List[str]:
    return [
        normalize_note(tok, ignore_modifiers)
        for tok in str(note_string).split()
        if tok.strip()
    ]


def load_music_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "note" not in df.columns or "dastgah" not in df.columns or "name" not in df.columns:
        raise ValueError(f"Unexpected columns in {path}. Expected name, note, dastgah.")
    df["note"] = df["note"].astype(str).str.strip()
    df["dastgah"] = df["dastgah"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    return df


def build_sequences(
    df: pd.DataFrame,
    ignore_modifiers: bool = False,
    use_dastgah: bool = False,
) -> List[List[str]]:
    sequences: List[List[str]] = []
    for _, row in df.iterrows():
        seq = tokenize_notes(row["note"], ignore_modifiers=ignore_modifiers)
        if use_dastgah:
            seq = [f"__DASTGAH_{normalize_note(row['dastgah'])}__"] + seq
        sequences.append(seq)
    return sequences

In [ ]:
# ---------------------------------------------------------------------
# N-gram LM with Lidstone smoothing
# ---------------------------------------------------------------------
class NGramLM:
    def __init__(self, n: int, alpha: float = 1.0):
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.alpha = float(alpha)
        self.counts: Counter = Counter()
        self.context_counts: Counter = Counter()
        self.vocab: List[str] = []

    def fit(self, sequences: Sequence[Sequence[str]]) -> "NGramLM":
        self.counts = Counter()
        self.context_counts = Counter()
        vocab = set()

        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            vocab.update(padded)
            for i in range(len(padded) - self.n + 1):
                ngram = tuple(padded[i : i + self.n])
                context = ngram[:-1]
                word = ngram[-1]
                self.counts[ngram] += 1
                self.context_counts[context] += 1

        vocab.discard("<s>")
        self.vocab = sorted(vocab)
        return self

    def prob(self, word: str, context: Sequence[str]) -> float:
        context = tuple(context)
        if self.n == 1:
            context = ()
        else:
            if len(context) < self.n - 1:
                context = ("<s>",) * (self.n - 1 - len(context)) + context
            elif len(context) > self.n - 1:
                context = context[-(self.n - 1):]

        V = len(self.vocab)
        c_ngram = self.counts[context + (word,)]
        c_context = self.context_counts[context]
        return (c_ngram + self.alpha) / (c_context + self.alpha * V)

    def perplexity(self, sequences: Sequence[Sequence[str]]) -> float:
        total_logprob = 0.0
        total_tokens = 0

        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            total_tokens += len(seq) + 1  # count EOS as well
            for i in range(self.n - 1, len(padded)):
                context = tuple(padded[i - self.n + 1 : i]) if self.n > 1 else ()
                total_logprob += math.log(self.prob(padded[i], context))
        return math.exp(-total_logprob / total_tokens)

    def next_distribution(self, context: Sequence[str], forbid_eos: bool = False) -> List[Tuple[str, float]]:
        context = list(context)
        if self.n > 1:
            if len(context) < self.n - 1:
                context = ["<s>"] * (self.n - 1 - len(context)) + context
            else:
                context = context[-(self.n - 1):]

        words = [w for w in self.vocab if w != "<s>"]
        if forbid_eos:
            words = [w for w in words if w != "</s>"]

        dist = [(w, self.prob(w, context)) for w in words]
        dist.sort(key=lambda x: x[1], reverse=True)
        return dist

    def beam_generate_fixed_length(
        self,
        prefix: Sequence[str],
        completion_len: int = TARGET_COMPLETION_LEN,
        beam_width: int = BEAM_WIDTH,
        block_triple_repeat: bool = True,
    ) -> List[str]:
        """
        Generate exactly `completion_len` notes after `prefix`.

        This method never generates EOS because the assignment asks for
        exactly 10 missing notes.
        """
        beams: List[Tuple[float, List[str]]] = [(0.0, list(prefix))]

        for _ in range(completion_len):
            candidates: List[Tuple[float, List[str]]] = []
            for logp, seq in beams:
                context = seq[-(self.n - 1):] if self.n > 1 else []
                dist = self.next_distribution(context, forbid_eos=True)

                for word, p in dist[: max(beam_width * 3, 10)]:
                    if block_triple_repeat and len(seq) >= 2 and word == seq[-1] == seq[-2]:
                        continue
                    new_seq = seq + [word]
                    new_logp = logp + math.log(p)
                    candidates.append((new_logp, new_seq))

            if not candidates:
                break

            # Length-normalized ranking over the generated completion only.
            def score(item: Tuple[float, List[str]]) -> float:
                logp, seq = item
                new_len = max(1, len(seq) - len(prefix))
                return logp / new_len

            candidates.sort(key=score, reverse=True)
            beams = candidates[:beam_width]

        best = max(beams, key=lambda x: x[0] / max(1, len(x[1]) - len(prefix)))[1]
        return best[len(prefix):]

In [ ]:
# ---------------------------------------------------------------------
# Alpha tuning
# ---------------------------------------------------------------------
def tune_alpha_loo(
    sequences: Sequence[Sequence[str]],
    n: int,
    alpha_grid: Sequence[float],
) -> Tuple[float, Dict[float, float]]:
    """
    Leave-one-out validation on sentence-level perplexity.
    Returns:
        best_alpha, {alpha: mean_validation_perplexity}
    """
    scores: Dict[float, float] = {}

    for alpha in alpha_grid:
        perps = []
        for i in range(len(sequences)):
            train = list(sequences[:i]) + list(sequences[i + 1 :])
            val = [sequences[i]]
            lm = NGramLM(n=n, alpha=alpha).fit(train)
            perps.append(lm.perplexity(val))
        scores[alpha] = float(np.mean(perps))

    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores


In [ ]:
# ---------------------------------------------------------------------
# Experiment runner
# ---------------------------------------------------------------------
def train_and_report(df: pd.DataFrame, ignore_modifiers: bool = False, use_dastgah: bool = False):
    sequences = build_sequences(df, ignore_modifiers=ignore_modifiers, use_dastgah=use_dastgah)

    print("=" * 88)
    print(f"ignore_modifiers={ignore_modifiers} | use_dastgah={use_dastgah}")
    print(f"Number of pieces: {len(sequences)}")
    print(f"Vocabulary size: {len(set(tok for seq in sequences for tok in seq))}")
    print()

    models = {}
    tuned = {}

    for n in NGRAM_ORDERS:
        best_alpha, alpha_scores = tune_alpha_loo(sequences, n, ALPHA_GRID)
        tuned[n] = (best_alpha, alpha_scores)
        model = NGramLM(n=n, alpha=best_alpha).fit(sequences)
        models[n] = model

        print(f"[n={n}] best alpha = {best_alpha}")
        for a in ALPHA_GRID:
            print(f"  alpha={a:>4}: val perplexity={alpha_scores[a]:.4f}")
        print(f"  train perplexity (best alpha) = {model.perplexity(sequences):.4f}")
        print()

    print("Perplexity for the required alpha values (1.0 and 0.5):")
    for n in NGRAM_ORDERS:
        p1 = NGramLM(n=n, alpha=1.0).fit(sequences).perplexity(sequences)
        p05 = NGramLM(n=n, alpha=0.5).fit(sequences).perplexity(sequences)
        print(f"  n={n}: alpha=1.0 -> {p1:.4f}, alpha=0.5 -> {p05:.4f}")
    print()

    print("Fixed-length completions (10 notes) for each prompt and each model:")
    for n in NGRAM_ORDERS:
        print(f"\n--- n={n}, alpha={tuned[n][0]} ---")
        model = models[n]
        for i, prefix in enumerate(PROMPTS, 1):
            completion = model.beam_generate_fixed_length(prefix, completion_len=TARGET_COMPLETION_LEN)
            print(f"{i}. {' '.join(prefix)} -> {' '.join(completion)}")

    return models, tuned


def main():
    df = load_music_csv(CSV_PATH)

    # Main run: as-is.
    train_and_report(df, ignore_modifiers=False, use_dastgah=False)

    # Second run: ignore b/k modifiers.
    train_and_report(df, ignore_modifiers=True, use_dastgah=False)

    # Optional: use dastgah conditioning.
    # This is helpful only if the target piece's dastgah is known at inference time.
    train_and_report(df, ignore_modifiers=True, use_dastgah=True)

    train_and_report(df, ignore_modifiers=False, use_dastgah=True)


if __name__ == "__main__":
    main()


ignore_modifiers=False | use_dastgah=False
Number of pieces: 41
Vocabulary size: 14

[n=1] best alpha = 2.0
  alpha= 0.1: val perplexity=12.0478
  alpha= 0.2: val perplexity=12.0356
  alpha= 0.3: val perplexity=12.0264
  alpha= 0.4: val perplexity=12.0186
  alpha= 0.5: val perplexity=12.0115
  alpha=0.75: val perplexity=11.9960
  alpha= 1.0: val perplexity=11.9825
  alpha= 1.5: val perplexity=11.9591
  alpha= 2.0: val perplexity=11.9392
  train perplexity (best alpha) = 10.8379

[n=2] best alpha = 1.0
  alpha= 0.1: val perplexity=9.7932
  alpha= 0.2: val perplexity=9.4689
  alpha= 0.3: val perplexity=9.3300
  alpha= 0.4: val perplexity=9.2549
  alpha= 0.5: val perplexity=9.2110
  alpha=0.75: val perplexity=9.1664
  alpha= 1.0: val perplexity=9.1658
  alpha= 1.5: val perplexity=9.2167
  alpha= 2.0: val perplexity=9.2954
  train perplexity (best alpha) = 7.5064

[n=3] best alpha = 0.5
  alpha= 0.1: val perplexity=11.9744
  alpha= 0.2: val perplexity=10.7137
  alpha= 0.3: val perplexity=1

In [4]:
# -*- coding: utf-8 -*-
"""
Google Colab (N-gram music LM).

What this script does:
1) loads music.csv
2) normalizes notes optionally (ignore b/k modifiers, keep uppercase notes)
3) trains unigram / bigram / trigram Lidstone language models
4) tunes alpha by leave-one-out validation
5) computes perplexity
6) generates completions for the 5 incomplete prompts
7) repeats the whole experiment with modifiers ignored
8) optionally uses the dastgah column as a conditioning token

Important note:
- Generation is done with a fixed target length (10 missing notes).
- To avoid degenerate loops like C C C C..., a small repeated-token blocking
  heuristic is included in beam search. This makes the filled sequences more
  useful for the assignment, while still being a pure n-gram decoder.
"""

from __future__ import annotations

from collections import Counter
from pathlib import Path
import math
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
CSV_PATH = "music.csv"
PROMPTS = [
    ["S", "S", "S", "S"],
    ["G", "C", "G", "C"],
    ["C", "B", "C", "D"],
    ["A", "Bb", "C", "D"],
    ["A", "S", "B", "Db"],
]
NGRAM_ORDERS = [1, 2, 3]
ALPHA_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]
TARGET_COMPLETION_LEN = 10   # the missing part in each sequence
BEAM_WIDTH = 8
RANDOM_SEED = 42


# ---------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------
def normalize_note(token: str, ignore_modifiers: bool = False) -> str:
    """
    Normalize a note token.

    - Keeps 'S' as silence.
    - When ignore_modifiers=True:
        * uppercases the token
        * strips trailing 'b' or 'k' modifier if present
          (examples: Ab -> A, Bb -> B, Ak -> A, Bk -> B)
    """
    t = str(token).strip()
    if not t:
        return t

    if ignore_modifiers:
        if t.lower() == "s":
            return "S"
        t = t.upper()
        if len(t) > 1 and t[-1] in {"B", "K"} and t[0] in "ABCDEFG":
            t = t[0]
    return t


def tokenize_notes(note_string: str, ignore_modifiers: bool = False) -> List[str]:
    return [
        normalize_note(tok, ignore_modifiers)
        for tok in str(note_string).split()
        if tok.strip()
    ]


def load_music_csv(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "note" not in df.columns or "dastgah" not in df.columns or "name" not in df.columns:
        raise ValueError(f"Unexpected columns in {path}. Expected name, note, dastgah.")
    df["note"] = df["note"].astype(str).str.strip()
    df["dastgah"] = df["dastgah"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    return df


def build_sequences(
    df: pd.DataFrame,
    ignore_modifiers: bool = False,
    use_dastgah: bool = False,
) -> List[List[str]]:
    sequences: List[List[str]] = []
    for _, row in df.iterrows():
        seq = tokenize_notes(row["note"], ignore_modifiers=ignore_modifiers)
        if use_dastgah:
            seq = [f"__DASTGAH_{normalize_note(row['dastgah'])}__"] + seq
        sequences.append(seq)
    return sequences


def normalize_prompt(prompt: Sequence[str], ignore_modifiers: bool = False) -> List[str]:
    return [normalize_note(tok, ignore_modifiers=ignore_modifiers) for tok in prompt]


# ---------------------------------------------------------------------
# N-gram LM with Lidstone smoothing
# ---------------------------------------------------------------------
class NGramLM:
    def __init__(self, n: int, alpha: float = 1.0):
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.alpha = float(alpha)
        self.counts: Counter = Counter()
        self.context_counts: Counter = Counter()
        self.vocab: List[str] = []

    def fit(self, sequences: Sequence[Sequence[str]]) -> "NGramLM":
        self.counts = Counter()
        self.context_counts = Counter()
        vocab = set()

        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            vocab.update(padded)
            for i in range(len(padded) - self.n + 1):
                ngram = tuple(padded[i : i + self.n])
                context = ngram[:-1]
                word = ngram[-1]
                self.counts[ngram] += 1
                self.context_counts[context] += 1

        vocab.discard("<s>")
        self.vocab = sorted(vocab)
        return self

    def prob(self, word: str, context: Sequence[str]) -> float:
        context = tuple(context)
        if self.n == 1:
            context = ()
        else:
            if len(context) < self.n - 1:
                context = ("<s>",) * (self.n - 1 - len(context)) + context
            elif len(context) > self.n - 1:
                context = context[-(self.n - 1):]

        V = len(self.vocab)
        c_ngram = self.counts[context + (word,)]
        c_context = self.context_counts[context]
        return (c_ngram + self.alpha) / (c_context + self.alpha * V)

    def perplexity(self, sequences: Sequence[Sequence[str]]) -> float:
        total_logprob = 0.0
        total_tokens = 0

        for seq in sequences:
            padded = ["<s>"] * (self.n - 1) + list(seq) + ["</s>"]
            total_tokens += len(seq) + 1  # count EOS as well
            for i in range(self.n - 1, len(padded)):
                context = tuple(padded[i - self.n + 1 : i]) if self.n > 1 else ()
                total_logprob += math.log(self.prob(padded[i], context))
        return math.exp(-total_logprob / total_tokens)

    def next_distribution(self, context: Sequence[str], forbid_eos: bool = False) -> List[Tuple[str, float]]:
        context = list(context)
        if self.n > 1:
            if len(context) < self.n - 1:
                context = ["<s>"] * (self.n - 1 - len(context)) + context
            else:
                context = context[-(self.n - 1):]

        words = [w for w in self.vocab if w != "<s>"]
        if forbid_eos:
            words = [w for w in words if w != "</s>"]

        dist = [(w, self.prob(w, context)) for w in words]
        dist.sort(key=lambda x: x[1], reverse=True)
        return dist

    def beam_generate_fixed_length(
        self,
        prefix: Sequence[str],
        completion_len: int = TARGET_COMPLETION_LEN,
        beam_width: int = BEAM_WIDTH,
        block_triple_repeat: bool = True,
    ) -> List[str]:
        """
        Generate exactly `completion_len` notes after `prefix`.

        This method never generates EOS because the assignment asks for
        exactly 10 missing notes.
        """
        beams: List[Tuple[float, List[str]]] = [(0.0, list(prefix))]

        for _ in range(completion_len):
            candidates: List[Tuple[float, List[str]]] = []
            for logp, seq in beams:
                context = seq[-(self.n - 1):] if self.n > 1 else []
                dist = self.next_distribution(context, forbid_eos=True)

                for word, p in dist[: max(beam_width * 3, 10)]:
                    if block_triple_repeat and len(seq) >= 2 and word == seq[-1] == seq[-2]:
                        continue
                    new_seq = seq + [word]
                    new_logp = logp + math.log(p)
                    candidates.append((new_logp, new_seq))

            if not candidates:
                break

            # Length-normalized ranking over the generated completion only.
            def score(item: Tuple[float, List[str]]) -> float:
                logp, seq = item
                new_len = max(1, len(seq) - len(prefix))
                return logp / new_len

            candidates.sort(key=score, reverse=True)
            beams = candidates[:beam_width]

        best = max(beams, key=lambda x: x[0] / max(1, len(x[1]) - len(prefix)))[1]
        return best[len(prefix):]


# ---------------------------------------------------------------------
# Alpha tuning
# ---------------------------------------------------------------------
def tune_alpha_loo(
    sequences: Sequence[Sequence[str]],
    n: int,
    alpha_grid: Sequence[float],
) -> Tuple[float, Dict[float, float]]:
    """
    Leave-one-out validation on sentence-level perplexity.
    Returns:
        best_alpha, {alpha: mean_validation_perplexity}
    """
    scores: Dict[float, float] = {}

    for alpha in alpha_grid:
        perps = []
        for i in range(len(sequences)):
            train = list(sequences[:i]) + list(sequences[i + 1 :])
            val = [sequences[i]]
            lm = NGramLM(n=n, alpha=alpha).fit(train)
            perps.append(lm.perplexity(val))
        scores[alpha] = float(np.mean(perps))

    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores


# ---------------------------------------------------------------------
# Experiment runner
# ---------------------------------------------------------------------
def train_and_report(df: pd.DataFrame, ignore_modifiers: bool = False, use_dastgah: bool = False):
    sequences = build_sequences(df, ignore_modifiers=ignore_modifiers, use_dastgah=use_dastgah)

    print("=" * 88)
    print(f"ignore_modifiers={ignore_modifiers} | use_dastgah={use_dastgah}")
    print(f"Number of pieces: {len(sequences)}")
    print(f"Vocabulary size: {len(set(tok for seq in sequences for tok in seq))}")
    print()

    models = {}
    tuned = {}

    for n in NGRAM_ORDERS:
        best_alpha, alpha_scores = tune_alpha_loo(sequences, n, ALPHA_GRID)
        tuned[n] = (best_alpha, alpha_scores)
        model = NGramLM(n=n, alpha=best_alpha).fit(sequences)
        models[n] = model

        print(f"[n={n}] best alpha = {best_alpha}")
        for a in ALPHA_GRID:
            print(f"  alpha={a:>4}: val perplexity={alpha_scores[a]:.4f}")
        print(f"  train perplexity (best alpha) = {model.perplexity(sequences):.4f}")
        print()

    print("Perplexity for the required alpha values (1.0 and 0.5):")
    for n in NGRAM_ORDERS:
        p1 = NGramLM(n=n, alpha=1.0).fit(sequences).perplexity(sequences)
        p05 = NGramLM(n=n, alpha=0.5).fit(sequences).perplexity(sequences)
        print(f"  n={n}: alpha=1.0 -> {p1:.4f}, alpha=0.5 -> {p05:.4f}")
    print()

    print("Fixed-length completions (10 notes) for each prompt and each model:")
    for n in NGRAM_ORDERS:
        print(f"\n--- n={n}, alpha={tuned[n][0]} ---")
        model = models[n]
        for i, prefix in enumerate(PROMPTS, 1):
            normalized_prefix = normalize_prompt(prefix, ignore_modifiers=ignore_modifiers)
            completion = model.beam_generate_fixed_length(normalized_prefix, completion_len=TARGET_COMPLETION_LEN)
            print(f"{i}. {' '.join(normalized_prefix)} -> {' '.join(completion)}")

    return models, tuned


def main():
    df = load_music_csv(CSV_PATH)

    # Main run: as-is.
    train_and_report(df, ignore_modifiers=False, use_dastgah=False)

    # Second run: ignore b/k modifiers.
    train_and_report(df, ignore_modifiers=True, use_dastgah=False)

    # Optional: use dastgah conditioning.
    # This is helpful only if the target piece's dastgah is known at inference time.
    train_and_report(df, ignore_modifiers=True, use_dastgah=True)


if __name__ == "__main__":
    main()


ignore_modifiers=False | use_dastgah=False
Number of pieces: 41
Vocabulary size: 14

[n=1] best alpha = 2.0
  alpha= 0.1: val perplexity=12.0478
  alpha= 0.2: val perplexity=12.0356
  alpha= 0.3: val perplexity=12.0264
  alpha= 0.4: val perplexity=12.0186
  alpha= 0.5: val perplexity=12.0115
  alpha=0.75: val perplexity=11.9960
  alpha= 1.0: val perplexity=11.9825
  alpha= 1.5: val perplexity=11.9591
  alpha= 2.0: val perplexity=11.9392
  train perplexity (best alpha) = 10.8379

[n=2] best alpha = 1.0
  alpha= 0.1: val perplexity=9.7932
  alpha= 0.2: val perplexity=9.4689
  alpha= 0.3: val perplexity=9.3300
  alpha= 0.4: val perplexity=9.2549
  alpha= 0.5: val perplexity=9.2110
  alpha=0.75: val perplexity=9.1664
  alpha= 1.0: val perplexity=9.1658
  alpha= 1.5: val perplexity=9.2167
  alpha= 2.0: val perplexity=9.2954
  train perplexity (best alpha) = 7.5064

[n=3] best alpha = 0.5
  alpha= 0.1: val perplexity=11.9744
  alpha= 0.2: val perplexity=10.7137
  alpha= 0.3: val perplexity=1

# Run3:

In [ ]:
import math
import re
from collections import Counter, defaultdict
import pandas as pd

# =========================
# 1) Load data
# =========================
# In Colab
CSV_PATH = "/content/music.csv"

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df["note"] = df["note"].astype(str).str.strip()
df["dastgah"] = df["dastgah"].astype(str).str.strip()

# =========================
# 2) Preprocessing
# =========================
def tokenize_notes(note_string, remove_modifiers=False):
    toks = note_string.split()
    if remove_modifiers:
        # Ab -> A, Bb -> B, Ak -> A, Dk -> D, Db -> D, ...
        toks = [re.sub(r"([A-GS])[bk]$", r"\1", t) for t in toks]
    return toks

songs_raw = [tokenize_notes(s, remove_modifiers=False) for s in df["note"]]
songs_clean = [tokenize_notes(s, remove_modifiers=True) for s in df["note"]]

# Fragments from the question
prefixes_raw = [
    ["S", "S", "S", "S"],
    ["G", "C", "G", "C"],
    ["C", "B", "C", "D"],
    ["A", "Bb", "C", "D"],
    ["A", "S", "B", "Db"],
]
prefixes_clean = [[re.sub(r"([A-GS])[bk]$", r"\1", t) for t in p] for p in prefixes_raw]

# =========================
# 3) N-gram model
# =========================
def build_ngram_counts(seqs, n):
    counts = Counter()
    context_counts = Counter()
    vocab = set()

    for seq in seqs:
        vocab.update(seq)

    for seq in seqs:
        padded = ["<s>"] * (n - 1) + seq
        for i in range(n - 1, len(padded)):
            hist = tuple(padded[i - n + 1:i]) if n > 1 else tuple()
            w = padded[i]
            counts[hist + (w,)] += 1
            context_counts[hist] += 1

    return counts, context_counts, sorted(vocab)

def lidstone_prob(counts, context_counts, vocab, hist, w, alpha):
    V = len(vocab)
    return (counts[hist + (w,)] + alpha) / (context_counts[hist] + alpha * V)

def sentence_logprob(seq, counts, context_counts, vocab, n, alpha):
    lp = 0.0
    history = ["<s>"] * (n - 1)
    for w in seq:
        hist = tuple(history[-(n - 1):]) if n > 1 else tuple()
        p = lidstone_prob(counts, context_counts, vocab, hist, w, alpha)
        lp += math.log(p)
        history.append(w)
    return lp

def corpus_perplexity(seqs, counts, context_counts, vocab, n, alpha):
    total_lp = 0.0
    total_tokens = 0
    for seq in seqs:
        total_lp += sentence_logprob(seq, counts, context_counts, vocab, n, alpha)
        total_tokens += len(seq)
    return math.exp(-total_lp / total_tokens)

def loocv_perplexity(seqs, n, alpha):
    vals = []
    for i in range(len(seqs)):
        train = [seqs[j] for j in range(len(seqs)) if j != i]
        test = [seqs[i]]
        counts, context_counts, vocab = build_ngram_counts(train, n)
        vals.append(corpus_perplexity(test, counts, context_counts, vocab, n, alpha))
    return sum(vals) / len(vals)

# =========================
# 4) Generation
# =========================
def generate_sequence(prefix, counts, context_counts, vocab, n, alpha, steps=10):
    """
    Greedy decoding: choose the most probable next token at each step.
    """
    seq = prefix[:]
    vocab = sorted(set(vocab) | set(prefix))  # keep unknown prefix tokens valid
    for _ in range(steps):
        hist = tuple(seq[-(n - 1):]) if n > 1 else tuple()
        denom = context_counts[hist] + alpha * len(vocab)

        probs = {}
        for w in vocab:
            probs[w] = (counts[hist + (w,)] + alpha) / denom

        # deterministic tie-break: highest probability, then lexicographic
        next_w = sorted(probs.items(), key=lambda x: (-x[1], x[0]))[0][0]
        seq.append(next_w)
    return seq

def run_experiment(seqs, prefixes, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    for n in [1, 2, 3]:
        print(f"\n--- {n}-gram ---")
        for alpha in [1.0, 0.5]:
            ppl = loocv_perplexity(seqs, n, alpha)
            print(f"alpha={alpha:.1f} | LOOCV perplexity = {ppl:.4f}")

        # Search a small alpha grid for the best value
        grid = [round(x, 2) for x in [0.1, 0.2, 0.3, 0.4, 0.5,
                                      0.6, 0.7, 0.8, 0.9, 1.0]]
        scored = []
        for alpha in grid:
            scored.append((alpha, loocv_perplexity(seqs, n, alpha)))
        best_alpha, best_ppl = min(scored, key=lambda x: x[1])
        print(f"Best alpha on grid: {best_alpha} | perplexity = {best_ppl:.4f}")

        # Train final model on all data with the selected alpha
        counts, context_counts, vocab = build_ngram_counts(seqs, n)

        print("\nGenerated completions (10 notes each):")
        for i, prefix in enumerate(prefixes, start=1):
            out = generate_sequence(prefix, counts, context_counts, vocab, n, best_alpha, steps=10)
            continuation = out[len(prefix):]
            print(f"{i}. prefix={' '.join(prefix)}")
            print(f"   continuation={' '.join(continuation)}")
            print(f"   full={' '.join(out)}")

# =========================
# 5) Raw-data experiment
# =========================
run_experiment(songs_raw, prefixes_raw, "RAW DATA")

# =========================
# 6) Modifier-stripped experiment
# =========================
run_experiment(songs_clean, prefixes_clean, "MODIFIERS REMOVED (b/k stripped)")

# =========================
# 7) Optional: effect of dastgah
# =========================
# If a song's dastgah is known, you can train a separate n-gram per dastgah
# and compare prefix likelihoods. This is often useful, but labels are imbalanced
# and some rows have '-' so it should be used carefully.

def build_group_models(seqs, labels, n):
    groups = defaultdict(list)
    for seq, lab in zip(seqs, labels):
        if lab != "-":
            groups[lab].append(seq)
    models = {}
    for lab, gseqs in groups.items():
        models[lab] = build_ngram_counts(gseqs, n)
    return models

def prefix_logprob(prefix, counts, context_counts, vocab, n, alpha):
    history = ["<s>"] * (n - 1)
    lp = 0.0
    for w in prefix:
        hist = tuple(history[-(n - 1):]) if n > 1 else tuple()
        p = lidstone_prob(counts, context_counts, vocab, hist, w, alpha)
        lp += math.log(p)
        history.append(w)
    return lp

# Example: rank dastgah models for a given prefix (optional)
models = build_group_models(songs_raw, df["dastgah"].tolist(), n=3)
for prefix in prefixes_raw:
    scores = []
    for lab, (counts, context_counts, vocab) in models.items():
        scores.append((lab, prefix_logprob(prefix, counts, context_counts, vocab, 3, 0.5)))
    print(prefix, sorted(scores, key=lambda x: x[1], reverse=True))


RAW DATA

--- 1-gram ---
alpha=1.0 | LOOCV perplexity = 11.1494
alpha=0.5 | LOOCV perplexity = 11.1778
Best alpha on grid: 1.0 | perplexity = 11.1494

Generated completions (10 notes each):
1. prefix=S S S S
   continuation=C C C C C C C C C C
   full=S S S S C C C C C C C C C C
2. prefix=G C G C
   continuation=C C C C C C C C C C
   full=G C G C C C C C C C C C C C
3. prefix=C B C D
   continuation=C C C C C C C C C C
   full=C B C D C C C C C C C C C C
4. prefix=A Bb C D
   continuation=C C C C C C C C C C
   full=A Bb C D C C C C C C C C C C
5. prefix=A S B Db
   continuation=C C C C C C C C C C
   full=A S B Db C C C C C C C C C C

--- 2-gram ---
alpha=1.0 | LOOCV perplexity = 8.4842
alpha=0.5 | LOOCV perplexity = 8.5264
Best alpha on grid: 0.9 | perplexity = 8.4812

Generated completions (10 notes each):
1. prefix=S S S S
   continuation=D C C C C C C C C C
   full=S S S S D C C C C C C C C C
2. prefix=G C G C
   continuation=C C C C C C C C C C
   full=G C G C C C C C C C C C C